# GAN Example — Generating Handwritten Digits (MNIST)

A **Generative Adversarial Network (GAN)** built with TensorFlow / Keras. This is a
**DCGAN** (Deep Convolutional GAN), the convolutional variant introduced by Radford
et al. (2015).

## The big idea

A GAN learns to *generate* brand-new data that looks like a training set — here, new
images of handwritten digits that were never in MNIST. It does this with a game between
two neural networks:

- **Generator** $G$ — takes a random noise vector $z$ (sampled from a Gaussian) and
  outputs a fake `28x28` image $G(z)$. It never sees real images directly; it learns
  *only* from the gradient signal the discriminator passes back.
- **Discriminator** $D$ — a binary classifier that receives an image and outputs how
  *real* it believes the image is. It is trained on a mix of real MNIST images and the
  generator's fakes.

The two are trained with **opposing objectives** (hence *adversarial*):

- $D$ tries to **maximize** its accuracy: output "real" for real images, "fake" for fakes.
- $G$ tries to **fool** $D$: produce fakes that $D$ labels as real.

Formally this is a minimax game over the value function

$$\min_G \max_D \; \mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1 - D(G(z)))].$$

As training proceeds, $G$ is pushed toward images indistinguishable from real ones. At
the ideal equilibrium $D$ can do no better than chance (≈0.5 everywhere) because the
fakes have become as convincing as the real data.

## How one training step works

1. Sample a batch of real images and a batch of random noise.
2. Generate fakes from the noise with $G$.
3. Show $D$ both batches and compute its loss (score reals high, fakes low).
4. Update $D$.
5. Compute $G$'s loss from how well its fakes fooled $D$, then update $G$.

Both updates happen on every batch, so the two networks co-evolve.

## Why GAN training looks different from normal training

Because both networks change every step, this is a *moving-target* optimization. The
loss curves do **not** steadily decrease the way they do in ordinary supervised
learning — they oscillate. **Judge progress by the generated samples, not the loss
numbers.** Failure modes to recognize:

- **Mode collapse** — $G$ produces only a few digit shapes, losing variety.
- **Discriminator dominance** — if $D$ gets too strong too fast, $G$ receives a
  vanishing gradient and stops improving. We mitigate this with matched learning rates
  and a deliberately modest $D$ (dropout, no batch-norm).

> Tested with `tensorflow==2.1` (matches `requirements.txt`). Runs on CPU, but a GPU is much faster.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print('TensorFlow', tf.__version__)

## 1. Load and prepare the data

We only need the images, not the labels — a GAN is **unsupervised**, it just learns what
the data distribution looks like.

**Why scale pixels to `[-1, 1]`?** The generator's final layer uses a `tanh` activation,
whose output range is exactly `[-1, 1]`. Matching the real data to that range means the
discriminator sees reals and fakes on the same scale — otherwise it could trivially tell
them apart by brightness alone. The transform is `(pixel - 127.5) / 127.5`, mapping
`0→-1` and `255→+1`.

We then wrap the array in a `tf.data.Dataset`, which `.shuffle()`s (so batches aren't
correlated) and `.batch()`es the data for efficient streaming during training.

In [ ]:
(train_images, _), (_, _) = tf.keras.datasets.mnist.load_data()

train_images = train_images.reshape(-1, 28, 28, 1).astype('float32')
train_images = (train_images - 127.5) / 127.5  # scale to [-1, 1]

BUFFER_SIZE = 60000
BATCH_SIZE = 256

train_dataset = (tf.data.Dataset
                 .from_tensor_slices(train_images)
                 .shuffle(BUFFER_SIZE)
                 .batch(BATCH_SIZE))

print('Training images:', train_images.shape)

## 2. The Generator

The generator maps a 100-dim noise vector into a `28x28x1` image. It does the opposite of
a normal CNN: instead of shrinking an image down to features, it **grows** a small
feature map up to a full image.

Walking through the layers:

- **`Dense` → `Reshape`** turns the flat noise vector into a small `7x7x256` feature map
  — a tiny, deep 'seed image'.
- **`Conv2DTranspose`** (a.k.a. *deconvolution* / *fractionally-strided convolution*) is
  the upsampling workhorse. A layer with `strides=2` roughly **doubles** the spatial
  size, so we go `7x7 → 14x14 → 28x28`. While growing the resolution we shrink the
  channel depth `256 → 128 → 64 → 1`.
- **`BatchNormalization`** stabilizes training by keeping activations well-scaled — this
  is one of the key tricks that makes deep GANs train at all.
- **`LeakyReLU`** (instead of plain ReLU) lets a small negative gradient through, which
  helps avoid 'dead' units and is the DCGAN-recommended activation.
- The final `tanh` squashes outputs to `[-1, 1]`, matching our scaled data.

Note `use_bias=False` on layers followed by batch-norm: the batch-norm's own shift
parameter makes a separate bias redundant.

In [ ]:
NOISE_DIM = 100

def make_generator():
    model = tf.keras.Sequential([
        layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(NOISE_DIM,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((7, 7, 256)),

        layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same',
                               use_bias=False, activation='tanh'),
    ])
    return model

generator = make_generator()
generator.summary()

In [ ]:
# Sanity check: an untrained generator just produces noise (its weights are random)
noise = tf.random.normal([1, NOISE_DIM])
generated_image = generator(noise, training=False)
plt.imshow(generated_image[0, :, :, 0], cmap='gray')
plt.title('Untrained generator output')
plt.axis('off');

## 3. The Discriminator

The discriminator is an ordinary CNN binary classifier: image in, one number out. It
uses strided convolutions to halve the resolution each layer (`28→14→7`) while growing
the channel depth, then flattens to a single `Dense` unit.

Two design choices worth calling out:

- **No final sigmoid** — the output is a raw *logit*. We pair it with a
  `from_logits=True` loss below, which is numerically more stable than applying a
  sigmoid and then taking the log.
- **`Dropout`** randomly zeros activations during training. In a GAN it acts as a
  handicap that keeps the discriminator from becoming *too* good *too* fast — which, as
  noted above, would starve the generator of useful gradients.

In [ ]:
def make_discriminator():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[28, 28, 1]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1),
    ])
    return model

discriminator = make_discriminator()
discriminator.summary()

## 4. Losses and optimizers

Both losses are **binary cross-entropy** on the discriminator's logits. The trick is
*which labels each network wishes for*:

- **Discriminator loss** = how wrong $D$ is on two batches: it should call reals `1` and
  fakes `0`. We sum the cross-entropy of `D(real) vs 1` and `D(fake) vs 0`. Minimizing
  this makes $D$ a better detector.
- **Generator loss** = cross-entropy of `D(fake) vs 1`. The generator *wants* the
  discriminator to output `1` ("real") for its fakes. This is the **non-saturating** GAN
  loss from the original paper — it gives the generator a strong gradient even early on
  when its fakes are easily caught.

We give each network its **own** Adam optimizer so their updates don't interfere. Equal
learning rates (`1e-4`) keep the two-player game balanced.

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

## 5. The training step

This is the heart of the GAN. In a single step we:

1. Draw fresh noise and let $G$ turn it into fake images.
2. Run $D$ on the **real** batch and on the **fake** batch.
3. Compute both losses inside **two separate `GradientTape`s** — one tracking the
   generator's variables, one tracking the discriminator's.
4. Ask each tape for the gradients of its loss w.r.t. *only its own* network's weights,
   and apply them.

Using two tapes is what keeps the two optimizations independent: the generator update
never touches discriminator weights and vice-versa. The `@tf.function` decorator
compiles this Python function into a static TensorFlow graph the first time it runs,
which makes every subsequent call much faster.

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
# A fixed noise seed so we can watch the SAME 16 samples improve epoch after epoch.
# (Reusing the same input makes the visual progress obvious.)
num_examples = 16
seed = tf.random.normal([num_examples, NOISE_DIM])

def plot_samples(model, test_input, epoch=None):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        # undo the [-1, 1] scaling for display: x * 0.5 + 0.5 -> [0, 1]
        plt.imshow(predictions[i, :, :, 0] * 0.5 + 0.5, cmap='gray')
        plt.axis('off')
    if epoch is not None:
        fig.suptitle(f'Epoch {epoch}')
    plt.show()

## 6. Train

We loop over the dataset for `EPOCHS` passes, calling `train_step` on every batch. Every
few epochs we render the fixed seed so you can watch the digits emerge from noise.

`EPOCHS = 30` is enough to get clearly recognisable digits. Lower it for a quick run;
raise it for sharper results. Remember: the printed losses will *oscillate* rather than
fall — that is normal and healthy for a GAN. What you want to see is the **samples**
improving.

In [ ]:
EPOCHS = 30

for epoch in range(1, EPOCHS + 1):
    g_loss = d_loss = 0.0
    for image_batch in train_dataset:
        g_loss, d_loss = train_step(image_batch)

    print(f'Epoch {epoch:3d}/{EPOCHS}  gen_loss={float(g_loss):.3f}  disc_loss={float(d_loss):.3f}')
    if epoch % 5 == 0 or epoch == 1:
        plot_samples(generator, seed, epoch)

## 7. Final generated digits

Now we feed **fresh** noise (not the fixed seed) to confirm the generator produces
varied, novel digits rather than memorising a handful of samples.

In [ ]:
plot_samples(generator, tf.random.normal([num_examples, NOISE_DIM]))

### Things to try

- Increase `EPOCHS` and watch the digits sharpen.
- Change `NOISE_DIM` or the layer widths and see how capacity affects quality.
- **Interpolate** in noise space: take two noise vectors and decode points along the
  line between them — the digits should morph smoothly, showing the space is continuous.
- Save the generator with `generator.save('gan_generator.h5')` and reload it to sample
  new digits without retraining.
- Compare this with the **autoencoder** example (`../autoencoder_example/`): both build a
  low-dimensional code that decodes to an image, but they learn it in very different ways.